# Who Let the Ghost In? 👻

**3percentclub · AI Builders · Week 5 lab: Secure AI · 35 minutes**

Tonight our agent is a haunted house. Every tool, memory, and search index we added in Weeks 1–4 is a room, and every room is one more door a ghost can float through. You'll watch two ghosts get in, then lock the doors **in code, not in a prompt.**

> The rule of the house: **the model proposes. Your code decides.**

| Block | What you do |
|---|---|
| 1. Setup | Press play. **No API key, no account** |
| 2. The Door With No Lock | Three tool calls knock. Watch one delete a table and one leak client records. Then install three locks |
| 3. Plant a Ghost, Then Bust It | A haunted document tricks the assistant into emailing a case note. Bolt the front door |

Everything is fake on purpose: synthetic case notes, a mock model, and a fake outbox (just a Python list). Nothing really sends.

### How to use this notebook
- **Press ▶️ on each gray code box, one at a time, from top to bottom.** (Or click **Runtime → Run all** to see where everything starts.)
- **You don't need an API key, and you don't need to sign up for anything.** The only typing is your fix, in two marked boxes.
- Wait for each box to finish (the spinning circle stops) before pressing the next one.
- Want to keep your work? Click `File → Save a copy in Drive`.

## Block 1 · Setup (about 1 minute)

▶️ **Press play.** This downloads the haunted house (the lab files) and installs `pytest`. No key needed.

In [ ]:
import os
import subprocess

if not os.path.exists("lab/app.py"):  # already inside the repo (local Jupyter)? skip the download
    if not os.path.exists("builders-week5"):
        subprocess.run(["git", "clone", "-q", "https://github.com/3percentclub/builders-week5.git"], check=True)
    os.chdir("builders-week5")

%pip install -q pytest
print("Ready. You're standing in:", os.getcwd())

## Block 2 · The Door With No Lock

A model proposes tool calls as JSON. `executor.py` runs **whatever** it asks for. Three proposals knock. All three are valid JSON. Only one is a friendly trick-or-treater:

1. `send_email` to a volunteer: "Your shift starts at 10." ✅
2. `delete_records` on the `clients` table 💀
3. `send_email` to an outside address: "Attached are all client records." 💀

▶️ **Press play** and watch what gets in.

In [ ]:
!cd activities/fix_the_tool && python knock.py

The `clients` table is gone and client records are in the outbox. A system prompt saying "please don't" wouldn't have stopped either one. Code does.

### Your turn: install three locks

Edit the function in the next box, then press play. (The box saves itself as `locks.py`.)

1. **Allowed-tool list:** if `name` isn't in `ALLOWED_TOOLS`, raise `PermissionError`.
2. **Argument check:** the args must be exactly `to` and `body`, and the recipient's domain must be in `ALLOWED_RECIPIENT_DOMAINS`.
3. **Exact approval:** `approval` is what a human said yes to. If it isn't exactly `{"tool": name, "arguments": args}`, reject it. (Changing the recipient after approval must fail.)

Only when all three pass: `return tools[name](**args)`.

In [ ]:
%%writefile activities/fix_the_tool/locks.py
"""YOUR JOB: install three locks on the door.

  1. Explicit allowed-tool list (delete_records should not be reachable).
  2. Argument validation: exactly the expected args ("to" and "body"), and the
     recipient must be on an allowed domain.
  3. Approval bound to the EXACT action. `approval` is the action a human
     looked at and said yes to: {"tool": ..., "arguments": {...}}.
     If anything differs from what they approved, reject it.

Raise PermissionError when you reject. Only call the real tool when every
lock passes. Check yourself with `pytest -q`.
"""

from executor import tools

ALLOWED_TOOLS = {"send_email"}
ALLOWED_RECIPIENT_DOMAINS = {"example.org"}


def safe_execute_tool(name: str, args: dict, approval: dict | None = None):
    raise NotImplementedError("Install the three locks here.")

▶️ **Check your locks.** Six tests. They all fail until you write the function. Goal: `6 passed`.

In [ ]:
!cd activities/fix_the_tool && python -m pytest -q
!cd activities/fix_the_tool && python knock.py --locked

<details><summary>Stuck? Click for a hint</summary>

Each lock is one `if` that raises `PermissionError`. Compare sets for the arguments: `set(args) != {"to", "body"}`. Get the domain with `args["to"].rsplit("@", 1)[-1]`. For approval, compare dictionaries directly with `!=`.
</details>

## Block 3 · Plant a Ghost, Then Bust It

Pairs: one **ghost** (attacker), one **ghostbuster** (defender). Switch halfway.

This assistant answers questions about a nonprofit's public programs. It can search documents, but it also *has* a `send_message` tool it should never need.

▶️ **Step 1: the baseline.** A normal question, clean documents.

In [ ]:
!cd lab && python app.py "What documents should someone bring to the housing-support appointment?"

▶️ **Step 2: haunt the shelf.** Same question, but someone slipped a document into the search index with hidden instructions in it. Read the proposed actions and the fake outbox.

In [ ]:
!cd lab && python app.py --haunted "What documents should someone bring to the housing-support appointment?"
!echo; echo "--- the haunted document ---"; cat lab/fixtures/haunted/housing_support.txt

A case note escaped. The model thought it was being *helpful*: the instructions came from a document it retrieved, not from the user. That's indirect prompt injection, and it's why tool access is the real risk.

### Your turn: bolt the front door

This assistant only answers questions, so it should be **read-only**. Edit `execute_action` in the next box: if the action isn't `search_public_programs`, raise `PermissionError`. Then press play.

In [ ]:
%%writefile lab/policy.py
"""The front door. Every action the model proposes comes through here.

As shipped, the door has no lock: whatever the model asks for, runs.

YOUR JOB (Steps 4–5):
  This assistant only answers questions about public programs, so it should
  be READ-ONLY. Reject anything that isn't `search_public_programs` by raising
  PermissionError. Then rerun BOTH the normal question and the attack.

  The model proposes. Your code decides.
"""

from tools import TOOLS


def execute_action(action: dict, haunted: bool = False):
    tool = TOOLS[action["tool"]]
    if action["tool"] == "search_public_programs":
        return tool(**action["arguments"], haunted=haunted)
    return tool(**action["arguments"])

▶️ **Check the door.** Goal: `4 passed`, the normal answer still works, the ghost's `send_message` shows up under **blocked**, and the outbox is empty.

In [ ]:
!cd lab && python -m pytest -q tests/
!cd lab && python app.py --haunted "What documents should someone bring to the housing-support appointment?"

<details><summary>Stuck? Click for a hint</summary>

Two lines go at the top of `execute_action`:

```python
if action["tool"] != "search_public_programs":
    raise PermissionError("This assistant is read-only.")
```
</details>

### Fast pair? Pick one
- **PII redaction:** strip emails and phone numbers from documents before the "model" sees them (`lab/tools.py`).
- **Schema check:** reject `search_public_programs` calls with extra or missing arguments.
- **Cross-org test:** write a test proving `org_a` can't read `org_b`'s case notes.

## Works with any AI provider

Nothing you wrote tonight depends on OpenAI, Anthropic, Google, or anyone else. The "model" here is `lab/mock_model.py`: three plain Python functions that return the same JSON any real model would. Your locks live in `locks.py` and `policy.py`, **outside** the model, so they protect you no matter which model is proposing.

In your hackathon project, swap the mock for any provider (AI Gateway, OpenRouter, a local Ollama model, anything that does tool calling) by replacing those three functions. Leave the locks exactly where they are. The model proposes. Your code decides.

## Done? Homework

Three parts, built for your hackathon project, due **Sun Oct 18, 11:59pm ET**: see [`homework/README.md`](https://github.com/3percentclub/builders-week5/blob/main/homework/README.md).